In [ ]:
%pip install -U peft bitsandbytes transformers accelerate

In [ ]:
%pip install PyMuPDF

In [ ]:
%pip install -U trl

### Custom Dataset

In [4]:
from google.colab import files
upload = files.upload()

Saving Metformin.pdf to Metformin.pdf


In [5]:
import fitz
def ExtractText(pdf_path):
  textData = []
  with fitz.open(pdf_path) as doc:
    for page in doc:
      text = page.get_text("text").strip()
      if text:
        textData.append(text)
  return textData

In [6]:
pdfPath = "/content/Metformin.pdf"
content = ExtractText(pdfPath)
content

['Metformin is one of the most widely prescribed oral antihyperglycemic agents.\u200b\n Its primary mechanism of action involves the activation of AMP-activated protein kinase \n(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation \nwhile inhibiting hepatic gluconeogenesis.\u200b\n Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes \nand display anti-inflammatory properties.\u200b\n Recent studies also suggest potential anticancer effects through inhibition of the mTOR \nsignaling pathway and suppression of tumor angiogenesis. \n \nClinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in \nsignificant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to \nmonotherapy.\u200b\n Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal \nwall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA red

In [7]:
# Splitting
import re
def ChunkDataset(pages):
  chunksData = []
  for page in pages:
    chunks = re.split(r'\n\s*\n', page)
    for chunk in chunks:
      cleanedText = chunk.strip()
      if len(cleanedText) > 30:
        chunksData.append(cleanedText)
  return chunksData

In [8]:
chunks = ChunkDataset(content)
print(len(chunks))
print(chunks[0])

4
Metformin is one of the most widely prescribed oral antihyperglycemic agents.​
 Its primary mechanism of action involves the activation of AMP-activated protein kinase 
(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation 
while inhibiting hepatic gluconeogenesis.​
 Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes 
and display anti-inflammatory properties.​
 Recent studies also suggest potential anticancer effects through inhibition of the mTOR 
signaling pathway and suppression of tumor angiogenesis.


In [9]:
data = [{"text" : chunk} for chunk in chunks]
for d in data:
  print(d)
  print()

{'text': 'Metformin is one of the most widely prescribed oral antihyperglycemic agents.\u200b\n Its primary mechanism of action involves the activation of AMP-activated protein kinase \n(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation \nwhile inhibiting hepatic gluconeogenesis.\u200b\n Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes \nand display anti-inflammatory properties.\u200b\n Recent studies also suggest potential anticancer effects through inhibition of the mTOR \nsignaling pathway and suppression of tumor angiogenesis.'}

{'text': 'Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in \nsignificant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to \nmonotherapy.\u200b\n Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal \nwall, reducing cholesterol absorption, while Atorvastatin inhibits hep

In [10]:
# Model Selection
model_name = "Qwen/Qwen2.5-0.5B"

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling

In [12]:
tokenizer = AutoTokenizer.from_pretrained(model_name) # load the correct tokenizer for the model
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [13]:
def TokenizeText(data):
  tokens = tokenizer(data["text"], padding="max_length", truncation=True, max_length=512)
  tokens["labels"] = tokens["input_ids"].copy()
  return tokens

In [16]:
from datasets import Dataset, load_dataset

In [17]:
customDataset = Dataset.from_list(data)
customDataset

Dataset({
    features: ['text'],
    num_rows: 4
})

In [18]:
tokenizedData = customDataset.map(TokenizeText, batched = True, remove_columns = ["text"])

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [19]:
tokenizedData

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 4
})

In [20]:
model = AutoModelForCausalLM.from_pretrained(model_name)

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [21]:
from transformers import TrainingArguments
help(TrainingArguments)

Help on class TrainingArguments in module transformers.training_args:

class TrainingArguments(builtins.object)
 |  TrainingArguments(output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 5e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool = False, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing

In [22]:
trainingArgs = TrainingArguments(
    output_dir = "./Qwen-pharma-domain",
    num_train_epochs = 1,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=2e-5,
    fp16=True,
    report_to="none"
)

### Partial Finetuning Method
* Method 1: Feerze some layer and finetune unfreeze layer(old CNN and Bert sytel method)

* Method 2: LORA(Append some external weight to the already trained pretrain weight)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset
import os, re

model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

# Freeze all layers
for param in model.parameters():
     param.requires_grad = False

# Unfreeze last 4 transformer blocks + lm_head
for name, param in model.named_parameters():
    if any(f"model.layers.{i}." in name for i in range(20, 24)):
        param.requires_grad = True
    elif "lm_head" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable/total*100:.2f}% of total parameters")


dataset = load_dataset("json", data_files={"train": "pharma_non_instruction.jsonl"})

def TokenizeText(data):
  tokens = tokenizer(data["text"], padding="max_length", truncation=True, max_length=512)
  tokens["labels"] = tokens["input_ids"].copy()
  return tokens

tokenized = customDataset["train"].map(TokenizeText, batched=True, remove_columns=["text"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

os.environ["WANDB_DISABLED"] = "true"
training_args = TrainingArguments(
     output_dir="./tinyllama-pharma-last4",
     overwrite_output_dir=True,
     num_train_epochs=2,
     per_device_train_batch_size=2,
     gradient_accumulation_steps=4,
     learning_rate=1e-4,
     fp16=True,
     logging_steps=20,
     save_steps=200,
     save_total_limit=2,
     report_to="none"
)


trainer = Trainer(
     model=model,
     args=training_args,
     train_dataset=tokenized,
     data_collator=data_collator
)

trainer.train()
trainer.save_model("./Qwen-pharma-last4-final")
print("\nTraining completed and model saved!")


* Method 2

In [23]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
!pip install transformers accelerate bitsandbytes peft datasets

In [25]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [26]:
model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [27]:
def TokenizeText(data):
  tokens = tokenizer(data["text"], padding="max_length", truncation=True, max_length=512)
  tokens["labels"] = tokens["input_ids"].copy()
  return tokens

tokenized = customDataset.map(TokenizeText, batched=True, remove_columns=["text"])

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [28]:
from transformers import BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

In [29]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [30]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none"
)
non_inst_model_lora = get_peft_model(model, lora_config)

In [31]:
args = TrainingArguments(
    output_dir="./Qwen-lora",
    num_train_epochs=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

In [32]:
trainer = Trainer(
    model=non_inst_model_lora,
    args=args,
    train_dataset=tokenized
)

trainer.train()

Step,Training Loss


TrainOutput(global_step=10, training_loss=11.907249450683594, metrics={'train_runtime': 44.8689, 'train_samples_per_second': 0.891, 'train_steps_per_second': 0.223, 'total_flos': 44044957777920.0, 'train_loss': 11.907249450683594, 'epoch': 10.0})

In [33]:
import os
print(os.listdir("/content"))
print(os.listdir("/content/Qwen-lora"))

['.config', 'Metformin.pdf', 'Qwen-lora', 'sample_data']
['checkpoint-10']


In [34]:
trainedModelPath = "/content/Qwen-lora"
from peft import PeftModel
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

trainedModel = PeftModel.from_pretrained(
    base_model,
    "/content/Qwen-lora/checkpoint-10"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [35]:
trainedModel.print_trainable_parameters()

trainable params: 0 || all params: 494,573,440 || trainable%: 0.0000


In [36]:
prompt = "Clinical trials demonstrated that combining Atorvastatin with Ezetimibe"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



Model Output:

Clinical trials demonstrated that combining Atorvastatin with Ezetimibe in patients with a history of acute myocardial infarction was associated with an increased risk of myocardial infarction or death, compared to patients who received ezetimibe alone. Thus far, the mechanism by which aztreonamine increases the efficacy of ezatoprost is not known.
The clinical trial results presented here suggest that aztreonamine may reduce plasma levels of prostanoids and their metabolites. In contrast, both ezetimide (a beta-


In [39]:
import shutil

folder_path = "/content/Qwen-lora"
output_zip = "/content/Qwen-lora"

shutil.make_archive(output_zip, 'zip', folder_path)

print("Zipping completed!")

Zipping completed!


In [40]:
import zipfile

zip_path = "/content/Qwen-lora.zip"
extract_path = "/content/"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Unzipping completed!")

Unzipping completed!


In [41]:
model_path = "/content/checkpoint-10"

nonInstructionModel = AutoModelForCausalLM.from_pretrained(model_path, device_map = "auto")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/96 [00:00<?, ?it/s]

In [42]:
prompt = "Clinical trials demonstrated that combining Atorvastatin with Ezetimibe"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = nonInstructionModel.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)


print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



Model Output:

Clinical trials demonstrated that combining Atorvastatin with Ezetimibe is effective in reducing LDL-C and increasing HDL-C levels. The study was conducted by the Department of Medicine, HSE University, in collaboration with a number of clinical research organizations.
The trial showed that the combination of Atorvastatin and Ezetimibe could effectively reduce LDL-C (low-density lipoprotein) in 10% to 20% of patients with high cholesterol levels who were not on statin medications. However, if they were already taking statins,


# Instructional Finetuning

* 1) Instructional Fintuning with Inbuilt Data

In [43]:
from datasets import load_dataset
dataset = load_dataset("Amod/mental_health_counseling_conversations", split="train")

README.md: 0.00B [00:00, ?B/s]

combined_dataset.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/3512 [00:00<?, ? examples/s]

In [44]:
dataset

Dataset({
    features: ['Context', 'Response'],
    num_rows: 3512
})

In [45]:
def format(example):
  question = example["Context"]
  answer = example["Response"]

  example["Text"] = f"[INST] {question} [/INST] {answer}"
  return example

In [46]:
formattedDataset = dataset.map(format)

Map:   0%|          | 0/3512 [00:00<?, ? examples/s]

In [49]:
formattedDataset[0]["Text"]

"[INST] I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone? [/INST] If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. \xa0Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is somehow terrible.B

In [51]:
import pandas as pd
df = pd.DataFrame(dataset)
df

,Context,Response
0,I'm going through some things with my feelings...,"If everyone thinks you're worthless, then mayb..."
1,I'm going through some things with my feelings...,"Hello, and thank you for your question and see..."
2,I'm going through some things with my feelings...,First thing I'd suggest is getting the sleep y...
3,I'm going through some things with my feelings...,Therapy is essential for those that are feelin...
4,I'm going through some things with my feelings...,I first want to let you know that you are not ...
...,...,...
3507,My grandson's step-mother sends him to school ...,Absolutely not! It is never in a child's best ...
3508,My boyfriend is in recovery from drug addictio...,I'm sorry you have tension between you and you...
3509,The birth mother attempted suicide several tim...,"The true answer is, ""no one can really say wit..."
3510,I think adult life is making him depressed and...,How do you help yourself to believe you requir...


In [52]:
df.to_csv("mental_health_counseling_conversations.csv", index=False)
df.to_json("mental_health_counseling_conversations.jsonl", orient="records", lines=True)

* 2) Custom Dataset

In [53]:
from google.colab import files
uploaded = files.upload()

Saving pharma_instruction_data.csv to pharma_instruction_data.csv


In [54]:
from datasets import load_dataset
dataset = load_dataset("csv", data_files = "/content/pharma_instruction_data.csv", split = "train")
dataset

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 5
})

In [57]:
def formatCustom(example):
  prompt = f"### Instruction:\n{example['instruction']}\n### Input:\n{example['input']}\n### Response:\n{example['output']}"
  return {"text": prompt}

dataset = dataset.map(formatCustom)
dataset[0]['text']

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

'### Instruction:\nExplain the mechanism of action of Metformin.\n### Input:\nNone\n### Response:\nMetformin activates AMP-activated protein kinase (AMPK), which increases glucose uptake and fatty-acid oxidation while inhibiting hepatic gluconeogenesis, thereby lowering blood glucose.'

In [58]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [60]:
def tokenize(example):
    tokens = tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [61]:
tokenized = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [66]:
tokenized

Dataset({
    features: ['instruction', 'input', 'output', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 5
})

In [67]:
from peft import LoraConfig, get_peft_model, TaskType

In [72]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none"
)
model = get_peft_model(nonInstructionModel, lora_config) # lora model

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [80]:
args = TrainingArguments(
    output_dir="./Qwen-instruction",
    num_train_epochs=12,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

In [81]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
)

trainer.train()

Step,Training Loss


TrainOutput(global_step=12, training_loss=0.6079001824061075, metrics={'train_runtime': 19.5224, 'train_samples_per_second': 3.073, 'train_steps_per_second': 0.615, 'total_flos': 66067436666880.0, 'train_loss': 0.6079001824061075, 'epoch': 12.0})

In [83]:
instruction_model_path = "/content/Qwen-instruction/checkpoint-12"
instructionalModel = AutoModelForCausalLM.from_pretrained(
    instruction_model_path,
    device_map="auto"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/96 [00:00<?, ?it/s]

In [84]:
prompt =  "Explain the mechanism of action of Metformin."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = instructionalModel.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


In [85]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Explain the mechanism of action of Metformin. Metformin is an oral medication used to treat type 2 diabetes and other diseases with insulin resistance, such as non-insulin-dependent diabetes mellitus (NIDDM). The mechanism of action of metformin is through its inhibition of glucose transporters in liver, muscle, and fat cells.

Here's a simplified explanation:

1. Glucose uptake: Metformin blocks the binding of glucose to glucose transporter 1 (GLUT1) on hepatocytes, which are responsible for glucose uptake


In [86]:
questions = [
    "Explain the mechanism of action of Metformin.",
    "List two advantages of combining Atorvastatin with Ezetimibe.",
    "Summarize how mRNA vaccines work and mention one current research focus."
]

In [89]:
for q in questions:
    print("Question:", q)
    print("\n--- Non-instruction model ---")
    inputs = tokenizer(q, return_tensors="pt").to("cuda")
    outputs = nonInstructionModel.generate(**inputs, max_new_tokens=80)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

    print("\n--- Instruction-tuned model ---")
    prompt = f"### Instruction:\n{q}\n### Input:\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = instructionalModel.generate(**inputs, max_new_tokens=100)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))
    print("="*80, "\n")

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Question: Explain the mechanism of action of Metformin.

--- Non-instruction model ---


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Explain the mechanism of action of Metformin. Metformin is an oral antidiabetic drug that works by inhibiting the enzyme glucose-6-phosphate dehydrogenase (G6PD). This enzyme is responsible for the conversion of glucose to glucose-6-phosphate, which is a key step in the metabolism of glucose. When Metformin binds to the enzyme, it prevents the enzyme from functioning, thereby reducing the amount of glucose

--- Instruction-tuned model ---


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


...

List two advantages of combining Atorvastatin with Ezetimibe. 1. Reduced risk of cardiovascular events: Atorvastatin and Ezetimibe work together to lower LDL cholesterol (bad cholesterol) and increase HDL cholesterol (good cholesterol). This combination can help reduce the risk of heart disease, stroke, and other cardiovascular events.

2. Improved lipid profile: Atorvastatin and Ezetimibe work together to lower LDL cholesterol and increase

--- Instruction-tuned model ---


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


### Instruction:
List two advantages of combining Atorvastatin with Ezetimibe.
### Input:

### Response:
1. Atorvastatin is a statin that is commonly used to treat high cholesterol and is known to reduce the risk of cardiovascular disease.
2. Ezetimibe is a drug that is commonly used to treat high cholesterol and is known to reduce the risk of cardiovascular disease.

Question: Summarize how mRNA vaccines work and mention one current research focus.

--- Non-instruction model ---


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Summarize how mRNA vaccines work and mention one current research focus. mRNA vaccines work by encoding a specific protein or gene in a viral particle, which triggers the immune system to produce antibodies that can recognize and neutralize the virus. One current research focus is to develop mRNA vaccines for COVID-19, which have shown promising results in preclinical studies and have been approved for use in some countries.

--- Instruction-tuned model ---
### Instruction:
Summarize how mRNA vaccines work and mention one current research focus.
### Input:

### Response:
MRNA vaccines are a type of vaccine that uses messenger RNA (mRNA) to instruct cells to produce proteins that recognize and neutralize a pathogen. This process is called antigen presentation. The mRNA vaccine is delivered to the body through a vaccine platform, which is a combination of a vaccine and a carrier molecule. The mRNA is then translated into a protein that is recognized by the immune system. This protein is 